# Riyadh Livability Index (RLI) — ML Engine
---
**Project:** Neighborhood DNA — The 15-Minute City Index  
**Input:** `Riyadh_Master_Dataset.csv` — 348K property rows × 29 columns  
**Pipeline:** Load → Aggregate → Density-Normalize → Log-Scale → Cluster-Impute → 4-Pillar RLI Score → Rank & Recommend  

**Outputs two systems:**
1. **Ranking** — City-wide livability ranking of all 176 neighborhoods (no user input).
2. **Recommendation** — Filter-first property search with K-Means cluster matching: category + price + rooms → qualifying neighborhoods ranked by RLI + price fit + cluster alignment.

**Formula:**

$$RLI_i = \left( \sum w_j \hat{x}_{ij} \right) \times \left[ 1 + \frac{H_i}{H_{max}} \right]$$

| Symbol | Meaning |
|---|---|
| $\hat{x}$ | MinMax-scaled log1p features (0–1) |
| $w$ | User-defined or default pillar weights |
| $H$ | Shannon Entropy $-\sum p \ln p$ across 4 pillars |
| $H_{max}$ | $\ln(4)$ — theoretical max diversity for 4 pillars |

**Combined Recommendation Score:**

$$Score = RLI \times w_{rli} + PriceScore \times w_{price} + ClusterFit \times w_{cluster}$$

## 1 | Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from math import pi

warnings.filterwarnings("ignore")

# ── Dark Theme (matches EDA / Clustering notebooks) ──
plt.rcParams.update({
    'figure.facecolor': '#0a0e27',
    'axes.facecolor': '#0a0e27',
    'axes.edgecolor': '#2a2f4e',
    'axes.labelcolor': '#c4c7d4',
    'text.color': '#c4c7d4',
    'xtick.color': '#8b8fa3',
    'ytick.color': '#8b8fa3',
    'grid.color': '#1a1f3e',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'font.size': 11,
    'figure.dpi': 120,
    'figure.figsize': (14, 6)
})

GOLD   = '#f0c05a'
CYAN   = '#4fc3f7'
CORAL  = '#ff6b6b'
MINT   = '#66bb6a'
PURPLE = '#ab47bc'
PALETTE = [GOLD, CYAN, CORAL, MINT, PURPLE, '#ff8a65', '#42a5f5', '#ef5350']

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Setup complete.')

Setup complete.


In [2]:
# ── Load Master Dataset ──
url = 'https://raw.githubusercontent.com/AbdulrahmanB-25/Machine_Learning_Competition/main/Riyadh_Master_Dataset.csv'

try:
    df_raw = pd.read_csv(url)
    print(f"Loaded from GitHub: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
except:
    df_raw = pd.read_csv('Riyadh_Master_Dataset.csv')
    print(f"Loaded locally: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

print(f"Unique neighborhoods: {df_raw['neighborhood'].nunique()}")
print(f"\nColumns: {df_raw.columns.tolist()}")
df_raw.head(3)

Loaded from GitHub: 348,704 rows × 29 columns
Unique neighborhoods: 176

Columns: ['neighborhood', 'property_id', 'price', 'area', 'category', 'total_rooms', 'lat', 'lng', 'dining_cafe', 'med_facilities', 'health_retail', 'fitness_care', 'edu_primary', 'edu_higher', 'religious', 'essential_retail', 'parks_green', 'sports_play', 'pedestrian', 'resort_rural_retreats', 'gov_civil', 'malls_shopping', 'Fiber_Available', 'FWA_Available', 'Mobile_Available', 'connectivity_score', 'bus_count', 'metro_count', 'neighborhood_area_km2']


,neighborhood,property_id,price,area,category,total_rooms,lat,lng,dining_cafe,med_facilities,health_retail,fitness_care,edu_primary,edu_higher,religious,essential_retail,parks_green,sports_play,pedestrian,resort_rural_retreats,gov_civil,malls_shopping,Fiber_Available,FWA_Available,Mobile_Available,connectivity_score,bus_count,metro_count,neighborhood_area_km2
0,2nd Industrial City,1,"4,500.00",160.00,1,6,24.56,46.86,46,2,1,0,1,0,0,7,3,3,0,0,9,4,0.00,1.00,1.00,2.00,53,0,33.71
1,2nd Industrial City,2,"17,000.00",150.00,1,4,24.55,46.87,46,2,1,0,1,0,0,7,3,3,0,0,9,4,0.00,1.00,1.00,2.00,53,0,33.71
2,2nd Industrial City,3,"1,500,000.00",750.00,2,0,24.56,46.86,46,2,1,0,1,0,0,7,3,3,0,0,9,4,0.00,1.00,1.00,2.00,53,0,33.71


## 2 | Constants & Configuration
All constants, category maps, pillar definitions, and feature groups — identical to `rli_engine.py`.

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════

EPSILON = 1e-9
H_MAX   = np.log(4)          # ln(4) — theoretical max Shannon entropy for 4 pillars
K_BEST  = 5                  # optimal cluster count from silhouette analysis

CATEGORY_MAP = {
    1:  'Apartment (Rent)',   2:  'Land',            3:  'Villa',
    4:  'Floor (Rent)',       5:  'Duplex (Rent)',    6:  'Apartment (Sale)',
    7:  'Commercial Land',    8:  'Office',           9:  'Building',
    10: 'Compound',           11: 'Farm',             13: 'Room',
    14: 'Shop',               15: 'Warehouse',        16: 'Commercial Building',
    17: 'Tower',              18: 'Camp',             19: 'Parking',
    20: 'Studio',             21: 'Chalet',           22: 'Duplex (Sale)',
    23: 'Rest House',         24: 'Palace',
}

# Categories where room filtering makes no sense
NO_ROOM_CATEGORIES = {
    'Land', 'Commercial Land', 'Farm', 'Rest House', 'Parking',
    'Warehouse', 'Camp', 'Shop',
}

# ── 4 RLI Pillars ──
PILLARS = {
    'Core': {
        'weight': 0.40,
        'features': ['med_facilities', 'edu_primary', 'essential_retail', 'religious'],
    },
    'Mobility': {
        'weight': 0.25,
        'features': ['bus_count', 'metro_count', 'pedestrian', 'connectivity_score'],
    },
    'Well-being': {
        'weight': 0.20,
        'features': ['dining_cafe', 'parks_green', 'sports_play', 'fitness_care'],
    },
    'Infrastructure': {
        'weight': 0.15,
        'features': ['Fiber_Available', 'gov_civil', 'malls_shopping', 'edu_higher'],
    },
}

ALL_RLI_FEATURES = sorted({f for p in PILLARS.values() for f in p['features']})

# Features that are raw counts and need density normalization (÷ km²)
COUNT_FEATURES = [
    'bus_count', 'metro_count', 'dining_cafe', 'med_facilities',
    'edu_primary', 'religious', 'essential_retail', 'parks_green',
    'sports_play', 'malls_shopping',
]

# ── Add category_name to df_raw ──
df_raw['category_name'] = df_raw['category'].map(CATEGORY_MAP)

print(f"RLI uses {len(ALL_RLI_FEATURES)} features across {len(PILLARS)} pillars:")
for pname, pinfo in PILLARS.items():
    print(f"  {pname:15s} ({pinfo['weight']:.0%}): {pinfo['features']}")

print(f"\nProperty Categories ({len(CATEGORY_MAP)}):")
for cat_id, label in sorted(CATEGORY_MAP.items()):
    count = (df_raw['category'] == cat_id).sum()
    if count > 0:
        print(f"  {cat_id:>2} → {label:<22s} ({count:>6,} properties)")

RLI uses 16 features across 4 pillars:
  Core            (40%): ['med_facilities', 'edu_primary', 'essential_retail', 'religious']
  Mobility        (25%): ['bus_count', 'metro_count', 'pedestrian', 'connectivity_score']
  Well-being      (20%): ['dining_cafe', 'parks_green', 'sports_play', 'fitness_care']
  Infrastructure  (15%): ['Fiber_Available', 'gov_civil', 'malls_shopping', 'edu_higher']

Property Categories (23):
   1 → Apartment (Rent)       (55,010 properties)
   2 → Land                   (81,838 properties)
   3 → Villa                  (124,911 properties)
   4 → Floor (Rent)           ( 8,107 properties)
   5 → Duplex (Rent)          (12,271 properties)
   6 → Apartment (Sale)       (28,910 properties)
   7 → Commercial Land        ( 7,587 properties)
   8 → Office                 ( 3,210 properties)
   9 → Building               (   643 properties)
  10 → Compound               ( 1,046 properties)
  11 → Farm                   (   107 properties)
  13 → Room             

## 3 | Core Function — `compute_rli()`
The single scoring function used by both the ranking and recommendation systems.  
Log-scales → MinMax normalizes → weighted pillar scores → Shannon entropy diversity boost → RLI 0–100.

In [4]:
def compute_rli(df_in: pd.DataFrame, pillar_weights: dict | None = None):
    """
    Compute Riyadh Livability Index for all neighborhoods in df_in.

    Parameters
    ----------
    df_in : DataFrame
        Neighborhood-level aggregated features (index = neighborhood name).
    pillar_weights : dict, optional
        Custom weights like {'Core': 0.5, 'Mobility': 0.2, ...}.
        Auto-normalized to sum to 1.0.

    Returns
    -------
    (df_result, scaler) : tuple
        df_result sorted by rank (best first), fitted MinMaxScaler.
    """
    pw = pillar_weights if pillar_weights else {p: v['weight'] for p, v in PILLARS.items()}
    # Normalize weights to sum to 1
    total_w = sum(pw.values())
    if total_w > 0:
        pw = {k: v / total_w for k, v in pw.items()}

    # Step 1: Log-scale then MinMax 0–1
    available = [f for f in ALL_RLI_FEATURES if f in df_in.columns]
    df_log = np.log1p(df_in[available])
    scaler = MinMaxScaler()
    df_sc = pd.DataFrame(
        scaler.fit_transform(df_log),
        columns=available,
        index=df_in.index,
    )

    # Step 2: Weighted pillar scores
    pillar_scores = {}
    for pname, pinfo in PILLARS.items():
        feats = [f for f in pinfo['features'] if f in df_sc.columns]
        w = pw.get(pname, 0)
        pillar_scores[pname] = df_sc[feats].mean(axis=1) * w

    pillar_df = pd.DataFrame(pillar_scores, index=df_in.index)
    weighted_sum = pillar_df.sum(axis=1)

    # Step 3: Shannon Entropy across 4 pillars
    pillar_props = pillar_df.div(pillar_df.sum(axis=1) + EPSILON, axis=0)
    H = -(pillar_props * np.log(pillar_props + EPSILON)).sum(axis=1)

    # Step 4: Diversity multiplier [1 + H / H_max]
    diversity_mult = 1 + (H / H_MAX)

    # Step 5: RLI = weighted_sum × diversity_multiplier
    raw_rli = weighted_sum * diversity_mult

    # Step 6: Normalize 0–100
    r_min, r_max = raw_rli.min(), raw_rli.max()
    rli_100 = ((raw_rli - r_min) / (r_max - r_min + EPSILON)) * 100

    # Assemble result
    result = df_in.copy()
    for pname in PILLARS:
        result[f'pillar_{pname}'] = pillar_scores[pname].values
    result['H_entropy']      = H.values
    result['diversity_mult'] = diversity_mult.values
    result['raw_rli']        = raw_rli.values
    result['RLI']            = rli_100.round(2).values
    result['rank']           = result['RLI'].rank(ascending=False, method='min').astype(int)

    return result.sort_values('rank'), scaler

print("compute_rli() defined ✓")

compute_rli() defined ✓


## 4 | Global Ranking Pipeline — `build_global_ranking()`
Aggregates all 348K properties → 176 neighborhoods, density-normalizes count features (÷ km²),
runs K-Means clustering, attaches PCA coordinates, and computes the city-wide RLI.

In [5]:
def build_global_ranking(df_raw: pd.DataFrame, pillar_weights: dict | None = None):
    """
    Aggregate all 348K properties → 176 neighborhoods, density-normalize,
    compute RLI, cluster (K-Means), and attach PCA coordinates.

    Returns
    -------
    dict with keys:
        'df_raw'     — original DataFrame (with category_name added)
        'df_global'  — aggregated neighborhood matrix
        'df_ranked'  — ranked DataFrame (index = neighborhood)
        'pca_2d'     — fitted PCA object
        'kmeans'     — fitted KMeans object
    """
    # Add category_name if missing
    if 'category_name' not in df_raw.columns:
        df_raw = df_raw.copy()
        df_raw['category_name'] = df_raw['category'].map(CATEGORY_MAP)

    # ── 1. City-wide aggregation ──
    df_global = df_raw.groupby('neighborhood').agg(
        neighborhood_area_km2=('neighborhood_area_km2', 'first'),
        property_count=('property_id', 'count'),
        median_price=('price', 'median'),
        median_area=('area', 'median'),
        lat=('lat', 'mean'),
        lng=('lng', 'mean'),
        n_categories=('category', 'nunique'),
        **{f: (f, 'mean') for f in ALL_RLI_FEATURES},
    )

    # ── 2. Density normalization (raw counts ÷ km²) ──
    for feat in COUNT_FEATURES:
        if feat in df_global.columns:
            df_global[feat] = df_global[feat] / df_global['neighborhood_area_km2']
    df_global = df_global.replace([np.inf, -np.inf], 0).fillna(0)

    # ── 3. K-Means clustering ──
    cluster_feats = [f for f in ALL_RLI_FEATURES if f in df_global.columns]
    X_std = (df_global[cluster_feats] - df_global[cluster_feats].mean()) / (df_global[cluster_feats].std() + EPSILON)
    kmeans = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
    df_global['km_cluster'] = kmeans.fit_predict(X_std)

    # ── 4. PCA for visualization ──
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_std)
    df_global['PC1'] = coords[:, 0]
    df_global['PC2'] = coords[:, 1]

    # ── 5. Compute RLI ──
    df_ranked, scaler = compute_rli(df_global, pillar_weights)

    return {
        'df_raw':    df_raw,
        'df_global': df_global,
        'df_ranked': df_ranked,
        'pca_2d':    pca,
        'kmeans':    kmeans,
        'scaler':    scaler,
    }

print("build_global_ranking() defined ✓")

build_global_ranking() defined ✓


## 5 | Recommendation Function — `recommend()`
Re-scores all neighborhoods with custom pillar weights. Returns re-ranked DataFrame with `match_pct`.

In [6]:
def recommend(df_ranked: pd.DataFrame, user_weights: dict | None = None, top_n: int = 20):
    """
    Re-score all neighborhoods with custom pillar weights.
    Returns re-ranked DataFrame with match_pct (% of top scorer).
    """
    df_re, _ = compute_rli(df_ranked, pillar_weights=user_weights)

    top_rli = df_re['RLI'].max()
    df_re['match_pct'] = (df_re['RLI'] / (top_rli + EPSILON) * 100).round(1)

    return df_re.head(top_n) if top_n else df_re

print("recommend() defined ✓")

recommend() defined ✓


## 6 | Property Search with K-Means Cluster Matching — `property_search()`
Filter-first property search **with K-Means cluster matching**:

1. Filter raw properties by category + price + rooms.
2. Identify qualifying neighborhoods.
3. **K-Means Step:** Score each cluster's average pillar profile against user weights → find best-fit cluster.
4. Boost neighborhoods in the best-fit cluster via `cluster_fit` score.
5. Rank by **combined score** = RLI × w_rli + Price Score × w_price + Cluster Fit × w_cluster.

This is where the ML (K-Means) actively drives the recommendation — it's not just scoring, it's matching the user to the neighborhood archetype that best fits their priorities.

In [7]:
def property_search(
    df_raw: pd.DataFrame,
    df_ranked: pd.DataFrame,
    category: int,
    min_price: float = 0,
    max_price: float = float('inf'),
    min_rooms: int = 0,
    pillar_weights: dict | None = None,
    price_weight: float = 0.20,
    cluster_weight: float = 0.15,
):
    """
    Filter-first property search with K-Means cluster matching.

    1. Filter raw properties by category + price + rooms.
    2. Identify qualifying neighborhoods.
    3. Re-score using RLI + price compatibility + K-Means cluster fit.

    K-Means Integration
    -------------------
    After filtering, the system computes the average pillar profile of each
    K-Means cluster among qualifying neighborhoods, then scores each cluster
    against the user's pillar weights. Neighborhoods in the best-fit cluster
    receive a boost, making the recommendation cluster-aware.

    Returns
    -------
    dict with keys:
        'results'              — DataFrame of ranked neighborhoods
        'properties_matched'   — int, how many raw properties survived
        'category_label'       — str
        'best_cluster'         — int, the cluster ID that best matches user priorities
        'cluster_scores'       — dict, {cluster_id: alignment_score}
    """
    cat_label = CATEGORY_MAP.get(category, f'Category {category}')

    # Ensure category_name exists
    if 'category_name' not in df_raw.columns:
        df_raw = df_raw.copy()
        df_raw['category_name'] = df_raw['category'].map(CATEGORY_MAP)

    # ── Step A: Strict filtering ──
    mask = df_raw['category'] == category
    mask &= df_raw['price'].between(min_price, max_price)

    if cat_label not in NO_ROOM_CATEGORIES and min_rooms > 0:
        mask &= df_raw['total_rooms'] >= min_rooms

    filtered = df_raw[mask]

    if len(filtered) == 0:
        return {
            'results': pd.DataFrame(),
            'properties_matched': 0,
            'category_label': cat_label,
            'best_cluster': -1,
            'cluster_scores': {},
        }

    # ── Step B: Neighborhood aggregation ──
    neigh_stats = filtered.groupby('neighborhood').agg(
        filtered_count=('property_id', 'count'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        avg_area=('area', 'mean'),
        avg_rooms=('total_rooms', 'mean'),
    ).round(1)

    qualifying = neigh_stats.index.tolist()
    df_score_input = df_ranked.loc[df_ranked.index.isin(qualifying)].copy()

    # ── Step C: Re-score with user weights ──
    df_scored, _ = compute_rli(df_score_input, pillar_weights=pillar_weights)

    # ── Step D: K-Means cluster matching ──
    # Score each cluster by how well its average pillar profile aligns with user weights
    pw = pillar_weights if pillar_weights else {p: v['weight'] for p, v in PILLARS.items()}
    total_w = sum(pw.values())
    if total_w > 0:
        pw = {k: v / total_w for k, v in pw.items()}

    cluster_scores = {}
    if 'km_cluster' in df_scored.columns:
        pillar_cols = [f'pillar_{p}' for p in PILLARS.keys()]
        available_pcols = [c for c in pillar_cols if c in df_scored.columns]

        for cid, grp in df_scored.groupby('km_cluster'):
            # Average pillar profile of this cluster
            cluster_profile = grp[available_pcols].mean()
            # Dot product with user weights = alignment score
            alignment = sum(
                cluster_profile.get(f'pillar_{p}', 0) * pw.get(p, 0)
                for p in PILLARS.keys()
            )
            cluster_scores[int(cid)] = round(alignment, 6)

        # Best cluster = highest alignment with user priorities
        best_cluster = max(cluster_scores, key=cluster_scores.get) if cluster_scores else -1

        # Normalize cluster scores to 0–100 for the boost
        cs_vals = list(cluster_scores.values())
        cs_min, cs_max = min(cs_vals), max(cs_vals)
        cs_range = cs_max - cs_min

        if cs_range > 0:
            df_scored['cluster_fit'] = df_scored['km_cluster'].map(
                lambda c: ((cluster_scores.get(c, 0) - cs_min) / cs_range) * 100
            )
        else:
            df_scored['cluster_fit'] = 100.0
    else:
        best_cluster = -1
        df_scored['cluster_fit'] = 0.0

    # ── Step E: Price compatibility score ──
    budget_mid = (min_price + max_price) / 2
    price_dist = np.abs(neigh_stats['avg_price'] - budget_mid)
    price_max_dist = price_dist.max()
    neigh_stats['price_score'] = (
        (1 - price_dist / price_max_dist) * 100 if price_max_dist > 0 else 100.0
    )

    # ── Step F: Merge & combined score (RLI + Price + Cluster Fit) ──
    overlap = neigh_stats.columns.intersection(df_scored.columns)
    df_result = df_scored.join(neigh_stats.drop(columns=overlap), how='inner')

    # Three-factor combined score
    rli_w = 1 - price_weight - cluster_weight
    rli_w = max(rli_w, 0)  # safety clamp
    df_result['combined_score'] = (
        df_result['RLI'] * rli_w
        + df_result['price_score'] * price_weight
        + df_result['cluster_fit'] * cluster_weight
    ).round(2)

    top_cs = df_result['combined_score'].max()
    df_result['match_pct'] = (
        (df_result['combined_score'] / (top_cs + EPSILON) * 100).round(1)
    )
    df_result['rank'] = df_result['combined_score'].rank(
        ascending=False, method='min'
    ).astype(int)
    df_result = df_result.sort_values('rank')

    return {
        'results': df_result,
        'properties_matched': len(filtered),
        'category_label': cat_label,
        'best_cluster': best_cluster,
        'cluster_scores': cluster_scores,
    }

print("property_search() defined ✓ (with K-Means cluster matching)")

property_search() defined ✓ (with K-Means cluster matching)


## 7 | Full Pipeline — `run_full_pipeline()`
Convenience function that loads the CSV and runs the entire pipeline. Used by the FastAPI backend on startup.

In [8]:
def run_full_pipeline(csv_path: str):
    """
    Load CSV → build global ranking → return pipeline dict.
    Keys: df_raw, df_global, df_ranked (=df_imputed=df_scored), pca_2d, kmeans, scaler.
    """
    df_raw = pd.read_csv(csv_path)
    pipe = build_global_ranking(df_raw)

    # Alias for backward compatibility with api.py
    pipe['df_scored']  = pipe['df_ranked']
    pipe['df_imputed'] = pipe['df_ranked']

    return pipe

print("run_full_pipeline() defined ✓")

run_full_pipeline() defined ✓


## 8 | Execute Pipeline
Run the full pipeline on the loaded dataset and display the global ranking.

In [9]:
# ── Run the global ranking pipeline ──
pipe = build_global_ranking(df_raw)
df_ranked = pipe['df_ranked']

print(f"✅ Global Ranking complete: {len(df_ranked)} neighborhoods scored.")
print(f"   Clusters: {df_ranked['km_cluster'].nunique()} (K-Means, k={K_BEST})")
print(f"   PCA: {pipe['pca_2d'].explained_variance_ratio_.round(4).tolist()}")
print(f"\nTop 10 Neighborhoods:")
print("=" * 70)
for _, r in df_ranked.head(10).iterrows():
    name = r.name if isinstance(r.name, str) else str(r.name)
    print(f"  #{int(r['rank']):>3}  {name:<30s}  RLI={r['RLI']:>6.2f}  Cluster={int(r['km_cluster'])}")

✅ Global Ranking complete: 176 neighborhoods scored.
   Clusters: 5 (K-Means, k=5)
   PCA: [0.1708, 0.1401]

Top 10 Neighborhoods:
  #  1  Al Aziziyah Dist.               RLI=100.00  Cluster=0
  #  2  Al Duraihemiyah Dist.           RLI= 97.56  Cluster=3
  #  3  Al Dhubbat Dist.                RLI= 96.25  Cluster=3
  #  4  Hiteen Dist.                    RLI= 89.97  Cluster=0
  #  5  West Umm Al Hamam Dist.         RLI= 89.40  Cluster=3
  #  6  Tuwaiq Dist.                    RLI= 83.01  Cluster=0
  #  7  Ar Rabwah Dist.                 RLI= 82.27  Cluster=3
  #  8  West Oraija Dist.               RLI= 81.80  Cluster=3
  #  9  Al Mansurah Dist.               RLI= 81.45  Cluster=3
  # 10  Al Ezdihar Dist.                RLI= 81.44  Cluster=3


## 9 | Demo — Custom Recommendation
Re-score all neighborhoods with mobility-heavy weights.

In [10]:
# ── Custom weights: mobility-focused user ──
custom_weights = {'Core': 0.15, 'Mobility': 0.50, 'Well-being': 0.25, 'Infrastructure': 0.10}

df_rec = recommend(df_ranked, user_weights=custom_weights, top_n=10)

print(f"Top 10 Neighborhoods (Mobility-Focused):")
print(f"  Weights: {custom_weights}")
print("=" * 70)
for _, r in df_rec.iterrows():
    name = r.name if isinstance(r.name, str) else str(r.name)
    print(f"  #{int(r['rank']):>3}  {name:<30s}  RLI={r['RLI']:>6.2f}  Match={r['match_pct']:>5.1f}%")

Top 10 Neighborhoods (Mobility-Focused):
  Weights: {'Core': 0.15, 'Mobility': 0.5, 'Well-being': 0.25, 'Infrastructure': 0.1}
  #  1  Al Aziziyah Dist.               RLI=100.00  Match=100.0%
  #  2  Hiteen Dist.                    RLI= 98.68  Match= 98.7%
  #  3  Al Dhubbat Dist.                RLI= 95.75  Match= 95.7%
  #  4  Al Duraihemiyah Dist.           RLI= 94.02  Match= 94.0%
  #  5  Al Dirah Dist.                  RLI= 91.51  Match= 91.5%
  #  6  Al Ezdihar Dist.                RLI= 90.23  Match= 90.2%
  #  7  Tuwaiq Dist.                    RLI= 87.32  Match= 87.3%
  #  8  Ar Rabwah Dist.                 RLI= 86.05  Match= 86.0%
  #  9  Al Sulaimaniyah Dist.           RLI= 84.29  Match= 84.3%
  # 10  Irqah                           RLI= 83.16  Match= 83.2%


## 10 | Demo — Property Search with K-Means Cluster Matching
Filter-first search: Villa under 2M SAR with 6+ rooms.  
The system uses K-Means to identify the best-fit neighborhood archetype and boosts those neighborhoods.

In [11]:
# ── Villa search with cluster matching ──
result = property_search(
    df_raw=pipe['df_raw'],
    df_ranked=df_ranked,
    category=3,             # Villa
    min_price=500_000,
    max_price=2_000_000,
    min_rooms=6,
    price_weight=0.20,
    cluster_weight=0.15,
)

df_res = result['results']
cat_label = result['category_label']
props = result['properties_matched']
best_cluster = result['best_cluster']
cluster_scores = result['cluster_scores']

print(f"\n{'═'*70}")
print(f"  Search: {cat_label} | 500,000 – 2,000,000 SAR | 6+ rooms")
print(f"{'─'*70}")
print(f"  {len(pipe['df_raw']):,} total properties")
print(f"  → {props:,} match your filters")
print(f"  → {len(df_res)} neighborhoods qualify")
print(f"  → {len(df_ranked) - len(df_res)} neighborhoods eliminated")
print(f"{'─'*70}")
print(f"  K-Means Best-Fit Cluster: {best_cluster}")
print(f"  Cluster Alignment Scores: {cluster_scores}")
print(f"{'═'*70}")

if len(df_res) > 0:
    print(f"\n  Top 10 Results:")
    print(f"  {'─'*65}")
    for _, r in df_res.head(10).iterrows():
        name = r.name.replace(' Dist.', '') if isinstance(r.name, str) else str(r.name)
        cf = r.get('cluster_fit', 0)
        print(f"  #{int(r['rank']):>2}  {name:<25s}  Match={r['match_pct']:>5.1f}%  "
              f"RLI={r['RLI']:>6.2f}  ClusterFit={cf:>5.1f}  "
              f"Avg={r.get('avg_price', 0):>12,.0f} SAR")


══════════════════════════════════════════════════════════════════════
  Search: Villa | 500,000 – 2,000,000 SAR | 6+ rooms
──────────────────────────────────────────────────────────────────────
  348,704 total properties
  → 67,889 match your filters
  → 149 neighborhoods qualify
  → 27 neighborhoods eliminated
──────────────────────────────────────────────────────────────────────
  K-Means Best-Fit Cluster: 3
  Cluster Alignment Scores: {0: np.float64(0.076279), 1: np.float64(0.03599), 2: np.float64(0.084178), 3: np.float64(0.084914)}
══════════════════════════════════════════════════════════════════════

  Top 10 Results:
  ─────────────────────────────────────────────────────────────────
  # 1  Al Dhubbat                 Match=100.0%  RLI=100.00  ClusterFit=100.0  Avg=   1,465,000 SAR
  # 2  Al Aziziyah                Match= 99.0%  RLI= 98.18  ClusterFit= 82.4  Avg=   1,143,606 SAR
  # 3  Al Duraihemiyah            Match= 97.4%  RLI= 93.86  ClusterFit=100.0  Avg=   1,091,674 SAR
 

## 11 | Export Global Ranking

In [12]:
# ── Export city-wide ranking ──
df_ranked.to_csv('Riyadh_Global_Ranking.csv', index=True)
print(f"Exported: 'Riyadh_Global_Ranking.csv'  ({len(df_ranked)} neighborhoods)")

try:
    from google.colab import files
    files.download('Riyadh_Global_Ranking.csv')
except ImportError:
    print("(Not in Colab — file saved locally)")

Exported: 'Riyadh_Global_Ranking.csv'  (176 neighborhoods)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12 | Key Findings

### How the Engine Works
1. **Input:** User specifies intent (Buy vs Rent), budget range, and lifestyle priorities.
2. **Filter:** System discards every property that doesn't match — only survivors continue.
3. **K-Means Matching:** System identifies which neighborhood archetype (cluster) best aligns with the user's pillar weights.
4. **Scoring:** Neighborhoods are ranked by a **three-factor combined score:**
   - **RLI** (livability) × 65%
   - **Price Score** (budget fit) × 20%
   - **Cluster Fit** (K-Means archetype match) × 15%
5. **Output:** Top 3 tailored recommendations with match percentage, livability score, and cluster info.

### How K-Means Drives the Recommendation
The K-Means clusters represent **neighborhood archetypes** discovered from the data. When a user sets their pillar weights (e.g., heavy on Mobility), the system:
1. Computes the average pillar profile of each cluster among qualifying neighborhoods.
2. Takes the dot product of each cluster profile with the user's weights.
3. The cluster with the highest alignment becomes the **best-fit cluster**.
4. All neighborhoods in that cluster receive a `cluster_fit` boost in the combined score.

This means the ML model is actively steering the recommendation — not just scoring.

### Architecture
| Component | File | Role |
|---|---|---|
| ML Engine | `rli_engine.py` | Single source of truth for all scoring + cluster matching |
| REST API | `api.py` | FastAPI backend — imports from `rli_engine.py` |
| Dashboard | `streamlit_app.py` | Streamlit UI — imports from `rli_engine.py` |
| Notebook | `RLI_Engine.ipynb` | This file — same code as `rli_engine.py`, runnable in Colab |